# Day 09. Exercise 00
# Regularization

## 0. Imports

In [74]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import joblib


## 1. Preprocessing

1. Read the file `dayofweek.csv` that you used in the previous day to a dataframe.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test`. Use the additional parameter `stratify`.

In [75]:
df = pd.read_csv("../data/dayofweek.csv")
df.head()
X = df.drop(columns=["dayofweek"])
y = df["dayofweek"]

In [76]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=21,
    stratify=y
)

## 2. Logreg regularization

### a. Default regularization

1. Train a baseline model with the only parameters `random_state=21`, `fit_intercept=False`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model


The result of the code where you trained and evaluated the baseline model should be exactly like this (use `%%time` to get the info about how long it took to run the cell):

```
train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64138   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
```

In [77]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = LogisticRegression(
        random_state=21, 
        fit_intercept=False
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")

    

train -  0.62819   |   valid -  0.59259
train -  0.64716   |   valid -  0.62963
train -  0.63479   |   valid -  0.57037
train -  0.65540   |   valid -  0.61481
train -  0.63314   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63591   |   valid -  0.62687
Average accuracy on crossval is 0.60239
Std is 0.02852
CPU times: user 104 ms, sys: 4.87 ms, total: 109 ms
Wall time: 109 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of penalty: `none`, `l1`, `l2` – you can change the values of solver too.

In [78]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = LogisticRegression(
        random_state=21,
        fit_intercept=False,
        solver="saga",
        max_iter=5000,
        C=np.inf,
        l1_ratio=0,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.66612   |   valid -  0.63704
train -  0.65787   |   valid -  0.65926
train -  0.66612   |   valid -  0.57778
train -  0.66529   |   valid -  0.62963
train -  0.66694   |   valid -  0.62222
train -  0.65952   |   valid -  0.57778
train -  0.65045   |   valid -  0.69630
train -  0.68425   |   valid -  0.61481
train -  0.66474   |   valid -  0.62687
train -  0.65651   |   valid -  0.60448
Average accuracy on crossval is 0.62462
Std is 0.03379
CPU times: user 14.9 s, sys: 80.6 ms, total: 15 s
Wall time: 15.3 s


In [79]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = LogisticRegression(
        random_state=21,
        fit_intercept=False,
        solver="saga",
        max_iter=5000,
        C=1.0,
        l1_ratio=1,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.63726   |   valid -  0.58519
train -  0.64221   |   valid -  0.61481
train -  0.62984   |   valid -  0.55556
train -  0.64386   |   valid -  0.60000
train -  0.63232   |   valid -  0.57778
train -  0.63644   |   valid -  0.57778
train -  0.63644   |   valid -  0.65926
train -  0.65622   |   valid -  0.57778
train -  0.64580   |   valid -  0.58955
train -  0.63839   |   valid -  0.62687
Average accuracy on crossval is 0.59646
Std is 0.02848
CPU times: user 2.65 s, sys: 11.2 ms, total: 2.66 s
Wall time: 2.66 s


In [80]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = LogisticRegression(
        random_state=21,
        fit_intercept=False,
        solver="saga",
        max_iter=5000,
        C=1.0,
        l1_ratio=0,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.62902   |   valid -  0.59259
train -  0.64633   |   valid -  0.62963
train -  0.63479   |   valid -  0.56296
train -  0.65622   |   valid -  0.61481
train -  0.63397   |   valid -  0.57778
train -  0.64056   |   valid -  0.59259
train -  0.64221   |   valid -  0.65926
train -  0.65952   |   valid -  0.56296
train -  0.64333   |   valid -  0.59701
train -  0.63674   |   valid -  0.62687
Average accuracy on crossval is 0.60165
Std is 0.02943
CPU times: user 473 ms, sys: 3 ms, total: 476 ms
Wall time: 476 ms


## 3. SVM regularization

### a. Default regularization

1. Train a baseline model with the only parameters `probability=True`, `kernel='linear'`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [81]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = SVC(
       probability=True,
       kernel="linear",
       random_state=21,
    )   
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")

    

/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.70486   |   valid -  0.65926
train -  0.69662   |   valid -  0.75556


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69415   |   valid -  0.62222
train -  0.70239   |   valid -  0.65185


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69085   |   valid -  0.65185
train -  0.68920   |   valid -  0.64444


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69250   |   valid -  0.72593
train -  0.70074   |   valid -  0.62222


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.69605   |   valid -  0.61940
train -  0.71087   |   valid -  0.63433
Average accuracy on crossval is 0.65871
Std is 0.04359
CPU times: user 1.7 s, sys: 9.01 ms, total: 1.71 s
Wall time: 1.74 s


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `C`.

In [82]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = SVC(
       probability=True,
       kernel="linear",
       random_state=21,
       C=0.1,
    )   
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.58120   |   valid -  0.55556
train -  0.57543   |   valid -  0.56296


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.57378   |   valid -  0.57037
train -  0.59275   |   valid -  0.57037


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.58120   |   valid -  0.54815
train -  0.57955   |   valid -  0.54815


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.57296   |   valid -  0.61481
train -  0.59192   |   valid -  0.54815


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.59967   |   valid -  0.52985
train -  0.57825   |   valid -  0.57463
Average accuracy on crossval is 0.56230
Std is 0.02177
CPU times: user 1.78 s, sys: 8 ms, total: 1.79 s
Wall time: 1.79 s


In [83]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = SVC(
       probability=True,
       kernel="linear",
       random_state=21,
       C=10,
    )   
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.75021   |   valid -  0.72593


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.77741   |   valid -  0.82963


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.77411   |   valid -  0.68889


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.78254   |   valid -  0.71642


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.78418   |   valid -  0.69403
Average accuracy on crossval is 0.72771
Std is 0.04417
CPU times: user 2.47 s, sys: 9.93 ms, total: 2.48 s
Wall time: 2.48 s


In [84]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = SVC(
       probability=True,
       kernel="linear",
       random_state=21,
       C=100,
    )   
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.78401   |   valid -  0.74815


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.79720   |   valid -  0.84444


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


...

train -  0.80462   |   valid -  0.73333


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.79819   |   valid -  0.70896


/Users/a1111/Desktop/DSB11_ML_Advanced_ID_1577654-1/venv/lib/python3.14/site-packages/sklearn/svm/_base.py:236: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


train -  0.79901   |   valid -  0.73881
Average accuracy on crossval is 0.75589
Std is 0.03550
CPU times: user 8.8 s, sys: 26.5 ms, total: 8.82 s
Wall time: 8.89 s


## 4. Tree

### a. Default regularization

1. Train a baseline model with the only parameter `max_depth=10` and `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [85]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=10,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.81039   |   valid -  0.74074
train -  0.77741   |   valid -  0.74074
train -  0.83347   |   valid -  0.70370
train -  0.79720   |   valid -  0.76296
train -  0.82440   |   valid -  0.75556
train -  0.80379   |   valid -  0.68889
train -  0.80709   |   valid -  0.76296
train -  0.80132   |   valid -  0.65926
train -  0.80807   |   valid -  0.75373
train -  0.80478   |   valid -  0.68657
Average accuracy on crossval is 0.72551
Std is 0.03562
CPU times: user 50.3 ms, sys: 1.65 ms, total: 51.9 ms
Wall time: 51.5 ms


### b. Optimizing regularization parameters

1. In the cells below try different values of the parameter `max_depth`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [86]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=3,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.49382   |   valid -  0.43704
train -  0.48887   |   valid -  0.48148
train -  0.50206   |   valid -  0.44444
train -  0.49629   |   valid -  0.49630
train -  0.48475   |   valid -  0.48889
train -  0.48969   |   valid -  0.48889
train -  0.48392   |   valid -  0.48148
train -  0.49052   |   valid -  0.40741
train -  0.48517   |   valid -  0.46269
train -  0.49176   |   valid -  0.42537
Average accuracy on crossval is 0.46140
Std is 0.02938
CPU times: user 39.4 ms, sys: 1.14 ms, total: 40.6 ms
Wall time: 40.3 ms


In [87]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=5,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.59522   |   valid -  0.53333
train -  0.56307   |   valid -  0.53333
train -  0.60181   |   valid -  0.55556
train -  0.59604   |   valid -  0.57037
train -  0.60264   |   valid -  0.57778
train -  0.57955   |   valid -  0.53333
train -  0.58368   |   valid -  0.54815
train -  0.59275   |   valid -  0.51111
train -  0.58237   |   valid -  0.56716
train -  0.60132   |   valid -  0.50000
Average accuracy on crossval is 0.54301
Std is 0.02423
CPU times: user 42.1 ms, sys: 1.36 ms, total: 43.5 ms
Wall time: 42.9 ms


In [88]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=15,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.95796   |   valid -  0.82222
train -  0.93075   |   valid -  0.83704
train -  0.95631   |   valid -  0.83704
train -  0.95301   |   valid -  0.86667
train -  0.95136   |   valid -  0.88889
train -  0.94724   |   valid -  0.82222
train -  0.95466   |   valid -  0.90370
train -  0.94971   |   valid -  0.87407
train -  0.95305   |   valid -  0.83582
train -  0.94316   |   valid -  0.85821
Average accuracy on crossval is 0.85459
Std is 0.02682
CPU times: user 52.9 ms, sys: 1.88 ms, total: 54.8 ms
Wall time: 54.5 ms


In [89]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=20,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.98846   |   valid -  0.86667
train -  0.99011   |   valid -  0.91111
train -  0.98681   |   valid -  0.85926
train -  0.98763   |   valid -  0.91111
train -  0.98928   |   valid -  0.88148
train -  0.98186   |   valid -  0.85926
train -  0.98846   |   valid -  0.91852
train -  0.99176   |   valid -  0.89630
train -  0.99094   |   valid -  0.88060
train -  0.98847   |   valid -  0.88060
Average accuracy on crossval is 0.88649
Std is 0.02075
CPU times: user 59.1 ms, sys: 2.21 ms, total: 61.3 ms
Wall time: 93.9 ms


In [90]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = DecisionTreeClassifier(
        max_depth=None,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  1.00000   |   valid -  0.85926
train -  1.00000   |   valid -  0.91852
train -  1.00000   |   valid -  0.86667
train -  1.00000   |   valid -  0.91111
train -  1.00000   |   valid -  0.88148
train -  1.00000   |   valid -  0.85185
train -  1.00000   |   valid -  0.92593
train -  1.00000   |   valid -  0.88148
train -  1.00000   |   valid -  0.88060
train -  1.00000   |   valid -  0.88060
Average accuracy on crossval is 0.88575
Std is 0.02374
CPU times: user 54.6 ms, sys: 1.73 ms, total: 56.4 ms
Wall time: 56.1 ms


## 5. Random forest

### a. Default regularization

1. Train a baseline model with the only parameters `n_estimators=50`, `max_depth=14`, `random_state=21`.
2. Use stratified K-fold cross-validation with `10` splits to evaluate the accuracy of the model.
3. The format of the result of the code where you trained and evaluated the baseline model should be similar to what you have got for the logreg.

In [91]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=14,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.96455   |   valid -  0.88148
train -  0.96208   |   valid -  0.91852
train -  0.96785   |   valid -  0.86667
train -  0.96455   |   valid -  0.89630
train -  0.96538   |   valid -  0.91111
train -  0.96538   |   valid -  0.88148
train -  0.97115   |   valid -  0.91852
train -  0.96867   |   valid -  0.85185
train -  0.97364   |   valid -  0.88060
train -  0.97941   |   valid -  0.86567
Average accuracy on crossval is 0.88722
Std is 0.02204
CPU times: user 537 ms, sys: 6.42 ms, total: 543 ms
Wall time: 542 ms


### b. Optimizing regularization parameters

1. In the new cells try different values of the parameters `max_depth` and `n_estimators`.
2. As a bonus, play with other regularization parameters trying to find the best combination.

In [92]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=10,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.85408   |   valid -  0.75556
train -  0.85903   |   valid -  0.82963
train -  0.89777   |   valid -  0.80741
train -  0.90107   |   valid -  0.82222
train -  0.88376   |   valid -  0.85185
train -  0.87799   |   valid -  0.75556
train -  0.87222   |   valid -  0.82963
train -  0.86480   |   valid -  0.74074
train -  0.89456   |   valid -  0.82090
train -  0.87644   |   valid -  0.73881
Average accuracy on crossval is 0.79523
Std is 0.04051
CPU times: user 490 ms, sys: 5.8 ms, total: 496 ms
Wall time: 496 ms


In [93]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=50,
        max_depth=20,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.99505   |   valid -  0.88889
train -  0.99835   |   valid -  0.94815
train -  0.99670   |   valid -  0.88148
train -  0.99753   |   valid -  0.93333
train -  0.99505   |   valid -  0.91852
train -  0.99670   |   valid -  0.88889
train -  0.99670   |   valid -  0.91852
train -  0.99753   |   valid -  0.91111
train -  0.99753   |   valid -  0.92537
train -  0.99588   |   valid -  0.87313
Average accuracy on crossval is 0.90874
Std is 0.02330
CPU times: user 577 ms, sys: 5.75 ms, total: 583 ms
Wall time: 583 ms


In [94]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=14,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.96950   |   valid -  0.87407
train -  0.96950   |   valid -  0.91111
train -  0.97197   |   valid -  0.87407
train -  0.96867   |   valid -  0.89630
train -  0.96867   |   valid -  0.90370
train -  0.96702   |   valid -  0.85926
train -  0.97444   |   valid -  0.91111
train -  0.96950   |   valid -  0.88148
train -  0.97117   |   valid -  0.88060
train -  0.97199   |   valid -  0.85821
Average accuracy on crossval is 0.88499
Std is 0.01869
CPU times: user 1.04 s, sys: 10.7 ms, total: 1.05 s
Wall time: 1.07 s


In [95]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.99670   |   valid -  0.89630
train -  0.99753   |   valid -  0.94815
train -  0.99670   |   valid -  0.89630
train -  0.99753   |   valid -  0.92593
train -  0.99670   |   valid -  0.91111
train -  0.99670   |   valid -  0.88889
train -  0.99670   |   valid -  0.91852
train -  0.99670   |   valid -  0.91111
train -  0.99918   |   valid -  0.92537
train -  0.99753   |   valid -  0.88060
Average accuracy on crossval is 0.91023
Std is 0.01925
CPU times: user 1.11 s, sys: 8.78 ms, total: 1.12 s
Wall time: 1.12 s


In [96]:
%%time
acc_list = []
cv = StratifiedKFold(n_splits=10)
for train_i, valid_i in cv.split(X_train, y_train):
    X_fold_train = X_train.iloc[train_i]
    y_fold_train = y_train.iloc[train_i]
    X_fold_valid = X_train.iloc[valid_i]
    y_fold_valid = y_train.iloc[valid_i]
    model = RandomForestClassifier(
        n_estimators=200,
        max_depth=20,
        random_state=21,
    )
    model.fit(X_fold_train, y_fold_train)
    train_pred = model.predict(X_fold_train)
    valid_pred = model.predict(X_fold_valid)
    train_ac = accuracy_score(y_fold_train, train_pred)
    valid_ac = accuracy_score(y_fold_valid, valid_pred)
    print(f"train -  {train_ac:.5f}   |   valid -  {valid_ac:.5f}")
    acc_list.append(valid_ac)
acc_list = np.array(acc_list)
print(f"Average accuracy on crossval is {np.mean(acc_list):.5f}")
print(f"Std is {np.std(acc_list):.5f}")


train -  0.99835   |   valid -  0.90370
train -  0.99753   |   valid -  0.94074
train -  0.99753   |   valid -  0.90370
train -  0.99835   |   valid -  0.93333
train -  0.99753   |   valid -  0.91111
train -  0.99670   |   valid -  0.88889
train -  0.99753   |   valid -  0.91852
train -  0.99835   |   valid -  0.91111
train -  0.99918   |   valid -  0.92537
train -  0.99753   |   valid -  0.88806
Average accuracy on crossval is 0.91245
Std is 0.01657
CPU times: user 2.17 s, sys: 14 ms, total: 2.18 s
Wall time: 2.19 s


## 6. Predictions

1. Choose the best model and use it to make predictions for the test dataset.
2. Calculate the final accuracy.
3. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your test dataset).
4. Save the model.

In [97]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=20,
    random_state=21,
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)
accuracy_score(y_test, y_pred)

0.9230769230769231

In [98]:
# Ищем ошибки 
errors = pd.DataFrame({
    "dayofweek": y_test,
    "error": y_test != y_pred
}
)

In [99]:
errors_by_day = errors.groupby("dayofweek")["error"].mean() * 100
print(errors_by_day)
worst_day_perc = errors_by_day.max()
worst_day = errors_by_day.idxmax()


dayofweek
0    25.925926
1    10.909091
2     6.666667
3     2.500000
4    14.285714
5     7.407407
6     2.816901
Name: error, dtype: float64


In [100]:
print(f"День с наибольшей долей ошибок: {worst_day}")
print(f"Процент ошибок: {worst_day_perc:.2f}%")

День с наибольшей долей ошибок: 0
Процент ошибок: 25.93%


In [101]:
joblib.dump(model, '../data/random_forest.joblib')

['../data/random_forest.joblib']